# StarCraft II — Guess the Matchup (starter)

Read `README.md` first. This notebook loads the data, trains a plain baseline, and scores it. Your job: beat it.

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)
train.head()

(496, 49) (333, 46)


,id,played_at,tournament,map,game_length,winner,matchup,p1_max_collection_rate,p2_max_collection_rate,p1_apm,...,p1_avg_collection_rate_gas,p1_avg_unspent_gas,p1_gas_lost,p1_gas_collected,p2_avg_collection_rate_gas,p2_avg_unspent_gas,p2_gas_lost,p2_gas_collected,p1_race,p2_race
0,7a41035d059e4e9683c57230c02ef7ba,2021-10-17 03:47:39+00:00,DH Masters Winter,Romanticide LE,957,2,PvP,4047,4064,342,...,630.6,449.0,5950,9229,595.3,275.4,4425,8764,Protoss,Protoss
1,559e7eeb399146219c8508d0c73f55a3,2021-09-30 23:11:17+00:00,DH Masters Winter,Lightshade LE,636,2,PvZ,4114,4848,446,...,583.9,284.6,3475,5349,627.2,206.9,2250,5638,Protoss,Zerg
2,e76c64d2d07545d0afc818c348acf333,2021-12-20 08:27:06+00:00,DH Masters Last Chance,Berlingrad - Ladder,373,1,TvP,2574,3095,370,...,229.0,119.7,156,1422,298.4,89.6,900,1718,Terran,Protoss
3,91c187a4d99c47758fd2a57a4ab4079c,2021-10-01 00:43:50+00:00,DH Masters Winter,Lightshade LE,825,2,PvP,3510,4310,286,...,381.2,211.8,2750,4390,570.5,262.7,3025,6207,Protoss,Protoss
4,3702aa54bc6d4988bd6793cae336b34b,2021-10-09 04:37:24+00:00,DH Masters Winter,Oxide LE,575,1,TvP,3308,2524,289,...,418.9,162.8,450,3516,311.7,161.7,800,2696,Protoss,Terran


## Scoring helper
Use it only at the end. For decisions, use cross-validation on `train`.

In [2]:
def score(pred_matchup):
    """pred_matchup: array of matchup strings, same order as test.csv"""
    labels = pd.read_csv("data/test_labels.csv").set_index("id").loc[test["id"], "matchup"]
    f1 = f1_score(labels, pred_matchup, average="macro")
    acc = accuracy_score(labels, pred_matchup)
    print(f"test macro-F1 = {f1:.4f}   accuracy = {acc:.4f}")
    return f1

## Baseline: XGBoost on the numeric columns

In [3]:
from xgboost import XGBClassifier

features = [c for c in test.columns if c.startswith(("p1_", "p2_"))] + ["game_length", "winner"]
classes = sorted(train["matchup"].unique())
y = train["matchup"].map({c: i for i, c in enumerate(classes)})

model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
model.fit(train[features], y)
pred = np.array(classes)[model.predict(test[features])]

## Your predictions
One row per test game, columns `id, matchup`.

In [4]:
submission = pd.DataFrame({"id": test["id"], "matchup": pred})
submission.head()

,id,matchup
0,8600ba65321a4b2b937f4919e25f535c,TvP
1,7a569c74abc14531bcea15c477949e0a,PvZ
2,084a958a02764cfea7fd5a9ab2a868da,PvP
3,ca91449c1dd34a74bf0a68e09ed32e38,ZvZ
4,acc01b5a602e4b0597149aa9a3dd6992,PvZ


In [5]:
score(submission['matchup'])

test macro-F1 = 0.5515   accuracy = 0.5616


0.551500433249828

## Ideas to try
- Look at the stats by race in `train` (`p1_race`, `p2_race`). What makes a Zerg player look different?
- The matchup ignores player order. Can you use that?
- Do you have to predict the matchup directly?